# Context-aware BERT – Nhận diện cảm xúc trong hội thoại (Nhóm 9)

**Yêu cầu:** Bật GPU trước khi chạy: `Runtime` → `Change runtime type` → **T4 GPU** → `Save`.

In [15]:
# 1. Clone repo và cài dependencies
%cd /content
!rm -rf /content/project_nhom9
!git clone https://github.com/trangdx2602/bert-context.git /content/project_nhom9
%cd /content/project_nhom9/Codebase

!pip install -r requirements.txt -q

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("LỖI: Chưa bật GPU trên Google Colab.")

/content
Cloning into '/content/project_nhom9'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 73 (delta 31), reused 59 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 4.21 MiB | 10.86 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/project_nhom9/Codebase
GPU: Tesla T4


## Upload dữ liệu MELD

Chạy cell bên dưới và upload 3 file CSV:
- `train_sent_emo.csv`
- `val_sent_emo.csv`
- `test_sent_emo.csv`

In [17]:
# 2. Upload file dữ liệu MELD
import os
from google.colab import files

os.makedirs('/content/project_nhom9/Documents', exist_ok=True)

print("Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/project_nhom9/Documents/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  Đã lưu: {dest}")

Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv


Saving test_sent_emo.csv to test_sent_emo.csv
Saving train_sent_emo.csv to train_sent_emo.csv
Saving val_sent_emo.csv to val_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/test_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/train_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/val_sent_emo.csv


## Huấn luyện – Ablation context window (k = 1, 3, 5)

Chạy tuần tự 3 cell bên dưới để so sánh ảnh hưởng của độ dài context.

In [18]:
# 3a. Huấn luyện với k=1 (1 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 1 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 1
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 5

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 1452.17it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.

In [19]:
# 3b. Huấn luyện với k=3 (3 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 3 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 3
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 5

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 925.33it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight          

In [20]:
# 3c. Huấn luyện với k=5 (5 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 5 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 5
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 5

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 954.40it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias            

## Đánh giá trên tập test

In [21]:
# 4a. Đánh giá k=1
!python evaluate.py --model bert_context --context_k 1 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1431.92it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=2, val_f1=0.5616
Evaluating: 100% 82/82 [00:04<00:00, 18.87it/s]

  Mo

In [22]:
# 4b. Đánh giá k=3
!python evaluate.py --model bert_context --context_k 3 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 894.31it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=2, val_f1=0.5630
Evaluating: 100% 82/82 [00:04<00:00, 18.77it/s]

  Mod

In [23]:
# 4c. Đánh giá k=5
!python evaluate.py --model bert_context --context_k 5 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1303.88it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=4, val_f1=0.5663
Evaluating: 100% 82/82 [00:04<00:00, 18.94it/s]

  Mo

## Tổng hợp kết quả

| Mô hình | Context k | Accuracy | Weighted F1 |
|---------|-----------|----------|-------------|
| Context-aware BERT | k=1 | 0.5471 | 0.5715 |
| Context-aware BERT | k=3 | 0.5429 | 0.5643 |
| Context-aware BERT | k=5 | 0.5678 | 0.5819 |

**Nhận xét:** k=5 đạt Weighted F1 cao nhất (0.5819), cho thấy ngữ cảnh dài hơn giúp mô hình dự đoán cảm xúc tốt hơn. Tuy nhiên xu hướng không đơn điệu — k=3 (0.5643) thấp hơn k=1 (0.5715), có thể do biến động trong quá trình huấn luyện hoặc MAX_LEN=128 bị truncate context ở k=3 nhiều hơn k=1. Nhìn chung, sử dụng context hội thoại (k≥1) cho kết quả cạnh tranh so với baseline không dùng context.